# Pipeline Bronze → Silver — TLC Trip Record Data

Este notebook ejecuta la transformación de la capa **Bronze** (datos crudos)
a la capa **Silver** (datos limpios, unificados y enriquecidos) dentro de la
arquitectura Medallion del proyecto.

**Lo que hace:**
- Lee los 4 tipos de vehículo (`yellow`, `green`, `fhv`, `fhvhv`) desde Bronze
- Unifica los esquemas a una estructura común (ver `src/silver/schemas.py`)
- Aplica las reglas de limpieza definidas en el EDA (ver `docs/decisiones_limpieza.md`)
- Agrega columnas calculadas (`duracion_min`, `hora`, `dia_semana`)
- Escribe a Silver particionado por `tipo_vehiculo / anio / mes`

**Diseño:** el procesamiento se hace archivo por archivo (no unión completa
en memoria) debido a la limitación de RAM del entorno (8GB). Cada función
incluye reintento automático ante fallos de memoria de Spark.

La lógica vive en `src/`, este notebook solo orquesta la ejecución.

## 1. Imports y verificación de configuración

Se cargan las funciones desde `src/` y se valida que la configuración
del proyecto (años, tipos de vehículo) sea la esperada antes de ejecutar
cualquier transformación.

In [1]:
import sys
sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session, ejecutar_con_reintento
from silver.transform import PROCESADORES
from config import ANIOS, MESES, TIPOS_VEHICULO

print("Imports OK")
print("Tipos:", TIPOS_VEHICULO)
print("Años:", ANIOS)

Imports OK
Tipos: ['yellow', 'green', 'fhv', 'fhvhv']
Años: [2023, 2024, 2025]


## 2. Ejecución del pipeline Bronze → Silver

Se itera por cada tipo de vehículo, año y mes. Por cada combinación:

1. Se verifica si ya fue procesado (`skip` automático, permite reanudar
   tras un fallo sin reprocesar lo ya hecho)
2. Se lee el archivo Bronze correspondiente
3. Se renombran columnas al esquema Silver unificado
4. Se aplican las reglas de limpieza específicas del tipo de vehículo
5. Se valida que el `pickup_datetime` corresponda al mes/año del archivo
   (corrige registros con fechas corruptas detectados en el EDA)
6. Se escribe a disco y se registra auditoría en `data/logs/auditoria_silver.csv`

Si Spark falla por memoria, `ejecutar_con_reintento()` reinicia la sesión
automáticamente y reintenta hasta 3 veces antes de marcar el mes como fallido.

In [3]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session, ejecutar_con_reintento
from silver.transform import PROCESADORES
from config import ANIOS, MESES, TIPOS_VEHICULO

spark = crear_spark_session()

for tipo in TIPOS_VEHICULO:
    print(f"\n{'='*50}")
    print(f"  PROCESANDO: {tipo.upper()}")
    print(f"{'='*50}")
    
    for anio in ANIOS:
        for mes in MESES:
            spark, exito = ejecutar_con_reintento(
                spark, PROCESADORES[tipo], anio, mes
            )
            if not exito:
                print(f"  [FALLO DEFINITIVO] {tipo} {anio}-{mes:02d}")

print("\n\nSilver completo!")
spark.stop()

2026-07-17 06:14:21,092 [INFO] SparkSession creada: 3.5.0



  PROCESANDO: YELLOW


2026-07-17 06:14:38,333 [INFO] [OK] yellow 2023-01: 3,066,766 -> 3,004,804 filas
2026-07-17 06:14:47,370 [INFO] [OK] yellow 2023-02: 2,913,955 -> 2,856,732 filas
2026-07-17 06:14:56,550 [INFO] [OK] yellow 2023-03: 3,403,766 -> 3,334,909 filas
2026-07-17 06:15:07,514 [INFO] [OK] yellow 2023-04: 3,288,250 -> 3,227,925 filas
2026-07-17 06:15:16,332 [INFO] [OK] yellow 2023-05: 3,513,649 -> 3,446,635 filas
2026-07-17 06:15:24,699 [INFO] [OK] yellow 2023-06: 3,307,234 -> 3,240,559 filas
2026-07-17 06:15:32,457 [INFO] [OK] yellow 2023-07: 2,907,108 -> 2,839,022 filas
2026-07-17 06:15:37,223 [INFO] [OK] yellow 2023-08: 2,824,209 -> 2,749,610 filas
2026-07-17 06:15:42,179 [INFO] [OK] yellow 2023-09: 2,846,722 -> 2,731,618 filas
2026-07-17 06:15:47,445 [INFO] [OK] yellow 2023-10: 3,522,285 -> 3,380,441 filas
2026-07-17 06:15:52,918 [INFO] [OK] yellow 2023-11: 3,339,715 -> 3,216,539 filas
2026-07-17 06:15:58,246 [INFO] [OK] yellow 2023-12: 3,376,567 -> 3,279,906 filas
2026-07-17 06:16:03,000 [INF


  PROCESANDO: GREEN


2026-07-17 06:18:09,144 [INFO] [OK] green 2023-01: 68,211 -> 64,080 filas
2026-07-17 06:18:09,869 [INFO] [OK] green 2023-02: 64,809 -> 61,084 filas
2026-07-17 06:18:10,619 [INFO] [OK] green 2023-03: 72,044 -> 67,686 filas
2026-07-17 06:18:11,488 [INFO] [OK] green 2023-04: 65,392 -> 61,426 filas
2026-07-17 06:18:12,304 [INFO] [OK] green 2023-05: 69,174 -> 64,799 filas
2026-07-17 06:18:13,098 [INFO] [OK] green 2023-06: 65,550 -> 61,262 filas
2026-07-17 06:18:13,896 [INFO] [OK] green 2023-07: 61,343 -> 56,998 filas
2026-07-17 06:18:14,645 [INFO] [OK] green 2023-08: 60,649 -> 56,359 filas
2026-07-17 06:18:15,369 [INFO] [OK] green 2023-09: 65,471 -> 61,172 filas
2026-07-17 06:18:16,142 [INFO] [OK] green 2023-10: 66,177 -> 61,564 filas
2026-07-17 06:18:16,809 [INFO] [OK] green 2023-11: 64,025 -> 59,749 filas
2026-07-17 06:18:17,574 [INFO] [OK] green 2023-12: 64,215 -> 59,978 filas
2026-07-17 06:18:18,329 [INFO] [OK] green 2024-01: 56,551 -> 52,820 filas
2026-07-17 06:18:19,088 [INFO] [OK] gr


  PROCESANDO: FHV


2026-07-17 06:18:36,053 [INFO] [OK] fhv 2023-01: 1,114,320 -> 229,674 filas
2026-07-17 06:18:37,619 [INFO] [OK] fhv 2023-02: 1,110,797 -> 227,577 filas
2026-07-17 06:18:38,981 [INFO] [OK] fhv 2023-03: 1,328,242 -> 277,050 filas
2026-07-17 06:18:40,511 [INFO] [OK] fhv 2023-04: 1,246,479 -> 257,232 filas
2026-07-17 06:18:41,907 [INFO] [OK] fhv 2023-05: 1,385,826 -> 314,111 filas
2026-07-17 06:18:43,425 [INFO] [OK] fhv 2023-06: 1,219,445 -> 289,777 filas
2026-07-17 06:18:44,770 [INFO] [OK] fhv 2023-07: 1,370,843 -> 311,015 filas
2026-07-17 06:18:46,224 [INFO] [OK] fhv 2023-08: 1,440,352 -> 354,396 filas
2026-07-17 06:18:47,696 [INFO] [OK] fhv 2023-09: 1,293,303 -> 345,588 filas
2026-07-17 06:18:49,057 [INFO] [OK] fhv 2023-10: 1,628,438 -> 376,711 filas
2026-07-17 06:18:50,286 [INFO] [OK] fhv 2023-11: 1,343,846 -> 185,340 filas
2026-07-17 06:18:51,969 [INFO] [OK] fhv 2023-12: 1,376,748 -> 227,918 filas
2026-07-17 06:18:53,464 [INFO] [OK] fhv 2024-01: 1,290,116 -> 258,484 filas
2026-07-17 0


  PROCESANDO: FHVHV


2026-07-17 06:20:28,798 [INFO] [OK] fhvhv 2023-01: 18,479,031 -> 18,168,469 filas
2026-07-17 06:21:13,549 [INFO] [OK] fhvhv 2023-02: 17,960,971 -> 17,680,156 filas
2026-07-17 06:22:03,246 [INFO] [OK] fhvhv 2023-03: 20,413,539 -> 20,034,607 filas
2026-07-17 06:22:51,695 [INFO] [OK] fhvhv 2023-04: 19,144,903 -> 19,102,037 filas
2026-07-17 06:23:41,771 [INFO] [OK] fhvhv 2023-05: 19,847,676 -> 19,810,101 filas
2026-07-17 06:24:30,691 [INFO] [OK] fhvhv 2023-06: 19,366,619 -> 19,329,922 filas
2026-07-17 06:25:18,417 [INFO] [OK] fhvhv 2023-07: 19,132,131 -> 19,105,728 filas
2026-07-17 06:25:40,446 [INFO] [OK] fhvhv 2023-08: 18,322,150 -> 18,298,764 filas
2026-07-17 06:26:05,192 [INFO] [OK] fhvhv 2023-09: 19,851,123 -> 19,837,742 filas
2026-07-17 06:26:43,873 [INFO] [OK] fhvhv 2023-10: 20,186,330 -> 20,177,714 filas
2026-07-17 06:27:19,911 [INFO] [OK] fhvhv 2023-11: 19,269,250 -> 19,253,646 filas
2026-07-17 06:27:41,427 [INFO] [OK] fhvhv 2023-12: 20,516,297 -> 20,505,691 filas
2026-07-17 06:28



Silver completo!


## 3. Verificación de resultados

Confirmamos que el Silver quedó completo y sin inconsistencias de fecha
antes de pasar a la capa Gold.

In [7]:
import os
from config import SILVER_PATH

total_meses = 0
for root, dirs, files in os.walk(SILVER_PATH):
    if "mes=" in root:
        total_meses += 1

print(f"Total de particiones mes encontradas: {total_meses}")

Total de particiones mes encontradas: 144


In [8]:
from spark_utils import crear_spark_session
spark = crear_spark_session()

2026-06-30 08:55:52,950 [INFO] SparkSession creada: 3.5.0


In [9]:
df_silver = spark.read.parquet("/home/jovyan/data/silver/trips/")
df_silver.createOrReplaceTempView("trips")
spark.sql("SELECT tipo_vehiculo, COUNT(*) FROM trips GROUP BY tipo_vehiculo").show()

+-------------+---------+
|tipo_vehiculo| count(1)|
+-------------+---------+
|       yellow|120733122|
|        fhvhv|713669341|
|          fhv| 11218792|
|        green|  1903467|
+-------------+---------+



In [7]:
import pandas as pd
from config import LOGS_PATH

aud = pd.read_csv(LOGS_PATH / "auditoria_silver.csv")
print(aud[aud["tipo_vehiculo"].isin(["yellow", "green"])].to_string())

2026-07-03 18:05:47,622 [INFO] NumExpr defaulting to 4 threads.


   tipo_vehiculo  anio  mes  filas_entrada  filas_salida  pct_retenido
0         yellow  2023    1        3066766       2989282         97.47
1         yellow  2023    1        3066766       2989282         97.47
2         yellow  2023    2        2913955       2841526         97.51
3         yellow  2023    3        3403766       3317032         97.45
4         yellow  2023    4        3288250       3209566         97.61
5         yellow  2023    5        3513649       3426929         97.53
6         yellow  2023    6        3307234       3221181         97.40
7         yellow  2023    7        2907108       2819724         96.99
8         yellow  2023    8        2824209       2729767         96.66
9         yellow  2023    9        2846722       2713343         95.31
10        yellow  2023   10        3522285       3357405         95.32
11        yellow  2023   11        3339715       3193557         95.62
12        yellow  2023   12        3376567       3254505         96.39
13    